In [0]:
%sql
CREATE OR REPLACE TABLE ecommerce.mart.fct_sales_flat AS
SELECT
  -- Grain identifiers
  oi.order_item_id,
  oi.order_id,
  o.order_date,

  -- Item-level metrics (safe to SUM at this grain)
  oi.quantity,
  oi.unit_price,
  oi.discount_percent,
  oi.discount_amount   AS item_discount_amount,
  oi.item_revenue,
  oi.unit_cost,
  oi.total_cost,
  oi.gross_profit,

  -- Product dimension
  p.product_id,
  p.product_name,
  p.brand,
  p.product_status,
  p.launch_date,
  cat.category_id,
  cat.category_name,
  cat.department,

  -- Order-level attributes (non-additive, safe to carry)
  o.order_status,
  CASE WHEN o.order_status = 'Completed' THEN TRUE ELSE FALSE END AS is_completed,
  o.shipping_method,

  -- Customer dimension
  c.customer_id,
  c.customer_segment,
  c.acquisition_channel,
  c.gender,
  c.city               AS customer_city,
  c.signup_date,

  -- Customer home region
  cust_r.region_id      AS customer_region_id,
  cust_r.region_name    AS customer_region_name,
  cust_r.market_type    AS customer_market_type,

  -- Shipping / fulfillment side
  o.region_id           AS shipping_region_id,
  ship_r.region_name    AS shipping_region_name,
  ship_r.market_type    AS shipping_market_type,
  s.store_id,
  s.store_name,
  s.store_type,

  -- Date dimension
  d.year,
  d.quarter,
  d.month,
  d.month_name,
  d.year_month,
  d.week,
  d.day_name,
  d.is_weekend,
  d.is_holiday,
  d.holiday_name

FROM ecommerce.clean.order_items      oi
JOIN ecommerce.clean.orders           o        ON oi.order_id     = o.order_id
JOIN ecommerce.clean.products         p        ON oi.product_id   = p.product_id
JOIN ecommerce.base.categories        cat      ON p.category_id   = cat.category_id
JOIN ecommerce.clean.customers        c        ON o.customer_id   = c.customer_id
LEFT JOIN ecommerce.base.regions      cust_r   ON c.region_id      = cust_r.region_id
LEFT JOIN ecommerce.base.regions      ship_r   ON o.region_id      = ship_r.region_id
LEFT JOIN ecommerce.base.stores       s        ON o.store_id       = s.store_id
LEFT JOIN ecommerce.base.date         d        ON o.order_date     = d.date;

In [0]:
%sql
SELECT is_completed, order_status, COUNT(*) AS row_count
FROM ecommerce.mart.fct_sales_flat
GROUP BY is_completed, order_status
ORDER BY is_completed DESC;
-- Expect: is_completed = TRUE only ever pairs with order_status = 'Completed'

### TEST 1 — Row count parity (mart should have exactly one row per order_item)

In [0]:
%sql
-- Expect: source_row_count = mart_row_count, row_diff = 0
SELECT
  (SELECT COUNT(*) FROM ecommerce.clean.order_items) AS source_row_count,
  (SELECT COUNT(*) FROM ecommerce.mart.fct_sales_flat) AS mart_row_count,
  (SELECT COUNT(*) FROM ecommerce.clean.order_items)
    - (SELECT COUNT(*) FROM ecommerce.mart.fct_sales_flat) AS row_diff;

### TEST 2 — Grain check (order_item_id must be unique)

In [0]:
%sql
-- Expect: 0 rows
SELECT order_item_id, COUNT(*) AS occurrences
FROM ecommerce.mart.fct_sales_flat
GROUP BY order_item_id
HAVING COUNT(*) > 1;

### TEST 3 — No fan-out risk from dimension tables (categories, regions, stores, date should each have unique keys)

In [0]:
%sql
-- Expect: 0 rows
SELECT 'categories' AS dim, category_id AS key_value, COUNT(*) AS dupes
FROM ecommerce.base.categories GROUP BY category_id HAVING COUNT(*) > 1
UNION ALL
SELECT 'regions', region_id, COUNT(*)
FROM ecommerce.base.regions GROUP BY region_id HAVING COUNT(*) > 1
UNION ALL
SELECT 'stores', store_id, COUNT(*)
FROM ecommerce.base.stores GROUP BY store_id HAVING COUNT(*) > 1
UNION ALL
SELECT 'date', date, COUNT(*)
FROM ecommerce.base.date GROUP BY date HAVING COUNT(*) > 1;

### TEST 4 — No orphaned rows dropped by inner joins

In [0]:
%sql
-- Expect: 0 rows (if >0, inner joins are silently excluding valid line items)
SELECT oi.order_item_id
FROM ecommerce.clean.order_items oi
LEFT JOIN ecommerce.mart.fct_sales_flat f ON oi.order_item_id = f.order_item_id
WHERE f.order_item_id IS NULL;

### TEST 5 — is_completed flag logic (should only be TRUE when order_status = 'Completed')

In [0]:
%sql
-- Expect: 0 rows
SELECT order_status, is_completed, COUNT(*) AS occurrences
FROM ecommerce.mart.fct_sales_flat
GROUP BY order_status, is_completed
HAVING (order_status = 'Completed' AND is_completed = FALSE)
    OR (order_status != 'Completed' AND is_completed = TRUE);

### TEST 6 — Revenue reconciliation (SUM(item_revenue) mart vs source)

In [0]:
%sql
-- Expect: revenue_diff = 0 (or negligible rounding, e.g. < 0.01)
SELECT
  (SELECT ROUND(SUM(item_revenue), 2) FROM ecommerce.clean.order_items) AS source_revenue,
  (SELECT ROUND(SUM(item_revenue), 2) FROM ecommerce.mart.fct_sales_flat) AS mart_revenue,
  (SELECT ROUND(SUM(item_revenue), 2) FROM ecommerce.clean.order_items)
    - (SELECT ROUND(SUM(item_revenue), 2) FROM ecommerce.mart.fct_sales_flat) AS revenue_diff;

### TEST 7 — Gross profit reconciliation

In [0]:
%sql
-- Expect: gp_diff = 0
SELECT
  (SELECT ROUND(SUM(gross_profit), 2) FROM ecommerce.clean.order_items) AS source_gp,
  (SELECT ROUND(SUM(gross_profit), 2) FROM ecommerce.mart.fct_sales_flat) AS mart_gp,
  (SELECT ROUND(SUM(gross_profit), 2) FROM ecommerce.clean.order_items)
    - (SELECT ROUND(SUM(gross_profit), 2) FROM ecommerce.mart.fct_sales_flat) AS gp_diff;

### TEST 8 — Excluded fields truly absent (order-level money fields should NOT be in this table) Run manually and confirm it errors with "column does not exist" — an error here is a PASS, not a failure.

In [0]:
%sql
-- Expect: query FAILS (column not found) — that failure is the pass condition
SELECT shipping_cost, tax_amount, order_total FROM ecommerce.mart.fct_sales_flat LIMIT 1;

### TEST 9 — Null profiling on key dimension columns

In [0]:
%sql
-- Expect: 0 for category_name/customer_segment/brand; some nulls in shipping_region/store/date
-- are acceptable (e.g. Pending/Cancelled orders may lack store_id) 
SELECT
  COUNT(*) AS total_rows,
  COUNT(*) - COUNT(category_name)        AS null_category,
  COUNT(*) - COUNT(customer_segment)     AS null_segment,
  COUNT(*) - COUNT(brand)                AS null_brand,
  COUNT(*) - COUNT(shipping_region_id)   AS null_shipping_region,
  COUNT(*) - COUNT(store_id)             AS null_store,
  COUNT(*) - COUNT(year)                 AS null_date_join
FROM ecommerce.mart.fct_sales_flat;

### TEST 10 — brand should never be literal NULL (clean.products already fills with 'Unknown')

In [0]:
%sql
-- Expect: 0 rows
SELECT COUNT(*) AS null_brand_rows
FROM ecommerce.mart.fct_sales_flat
WHERE brand IS NULL;

### TEST 11 — Referential sanity (product_id, customer_id, order_id all resolve to clean dimensions)

In [0]:
%sql
-- Expect: 0 rows for each check_type
SELECT 'product_id' AS check_type, f.product_id AS bad_key
FROM ecommerce.mart.fct_sales_flat f
LEFT JOIN ecommerce.clean.products p ON f.product_id = p.product_id
WHERE p.product_id IS NULL
UNION ALL
SELECT 'customer_id', f.customer_id
FROM ecommerce.mart.fct_sales_flat f
LEFT JOIN ecommerce.clean.customers c ON f.customer_id = c.customer_id
WHERE c.customer_id IS NULL
UNION ALL
SELECT 'order_id', f.order_id
FROM ecommerce.mart.fct_sales_flat f
LEFT JOIN ecommerce.clean.orders o ON f.order_id = o.order_id
WHERE o.order_id IS NULL;

### TEST 12 — Value range sanity on carried-over numeric fields

In [0]:
%sql
-- Expect: 0 rows
SELECT *
FROM ecommerce.mart.fct_sales_flat
WHERE quantity <= 0
   OR unit_price <= 0
   OR discount_percent < 0 OR discount_percent > 0.30
   OR item_revenue < 0;

### TEST 13 — Date join sanity (order_date should match the joined date-dimension row exactly)

In [0]:
%sql
-- Expect: 0 rows
SELECT f.order_id, f.order_date, d.date AS joined_date
FROM ecommerce.mart.fct_sales_flat f
LEFT JOIN ecommerce.base.date d ON f.order_date = d.date
WHERE d.date IS NULL OR f.order_date != d.date;

### TEST 14 — Distinct value spot-check on order_status

In [0]:
%sql
-- Expect: only 'Completed', 'Cancelled', 'Pending', 'Returned'
SELECT DISTINCT order_status FROM ecommerce.mart.fct_sales_flat;